<a href="https://colab.research.google.com/github/Simran1604/Neuromorphic-EuroSAT-SNN/blob/main/Neuromorphic_Eurosat.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
import torch.nn as nn
import torch.optim as optim

class StandardCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1, stride=2),
            nn.BatchNorm2d(32),
            nn.ReLU(),

            nn.Conv2d(32, 64, kernel_size=3, padding=1, stride=2),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.Conv2d(64, 128, kernel_size=3, padding=1, stride=2),
            nn.BatchNorm2d(128),
            nn.ReLU()
        )

        self.classifier = nn.Sequential(
            nn.Linear(128 * 8 * 8, 256),
            nn.ReLU(),
            nn.Linear(256, 10)
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1) # Flatten
        x = self.classifier(x)
        return x

cnn_net = StandardCNN().to(device)
print("Standard CNN initialized.")

Standard CNN initialized.


In [14]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(cnn_net.parameters(), lr=1e-3)

epochs = 15
print("Training Continuous ANN...")

for epoch in range(epochs):
    cnn_net.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for i, (data, targets) in enumerate(train_loader):
        data, targets = data.to(device), targets.to(device)

        optimizer.zero_grad()
        outputs = cnn_net(data)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()

    acc = 100. * correct / total
    print(f"Epoch {epoch+1}/{epochs} | Loss: {running_loss/len(train_loader):.4f} | ANN Accuracy: {acc:.2f}%")

Training Continuous ANN...
Epoch 1/15 | Loss: 0.9733 | ANN Accuracy: 65.37%
Epoch 2/15 | Loss: 0.6875 | ANN Accuracy: 75.24%
Epoch 3/15 | Loss: 0.5971 | ANN Accuracy: 78.44%
Epoch 4/15 | Loss: 0.5251 | ANN Accuracy: 81.11%
Epoch 5/15 | Loss: 0.4865 | ANN Accuracy: 82.88%
Epoch 6/15 | Loss: 0.4331 | ANN Accuracy: 84.89%
Epoch 7/15 | Loss: 0.3930 | ANN Accuracy: 86.12%
Epoch 8/15 | Loss: 0.3657 | ANN Accuracy: 87.08%
Epoch 9/15 | Loss: 0.3294 | ANN Accuracy: 88.42%
Epoch 10/15 | Loss: 0.3055 | ANN Accuracy: 89.06%
Epoch 11/15 | Loss: 0.2847 | ANN Accuracy: 89.88%
Epoch 12/15 | Loss: 0.2746 | ANN Accuracy: 90.58%
Epoch 13/15 | Loss: 0.2466 | ANN Accuracy: 91.25%
Epoch 14/15 | Loss: 0.2379 | ANN Accuracy: 91.75%
Epoch 15/15 | Loss: 0.2335 | ANN Accuracy: 91.66%


In [21]:
import snntorch as snn

snn_beta = 1.0
snn_thresh = 0.5

class ConvertedSNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1, stride=2),
            nn.BatchNorm2d(32),
            snn.Leaky(beta=snn_beta, threshold=snn_thresh),

            nn.Conv2d(32, 64, kernel_size=3, padding=1, stride=2),
            nn.BatchNorm2d(64),
            snn.Leaky(beta=snn_beta, threshold=snn_thresh),

            nn.Conv2d(64, 128, kernel_size=3, padding=1, stride=2),
            nn.BatchNorm2d(128),
            snn.Leaky(beta=snn_beta, threshold=snn_thresh)
        )

        self.classifier = nn.Sequential(
            nn.Linear(128 * 8 * 8, 256),
            snn.Leaky(beta=snn_beta, threshold=snn_thresh),
            nn.Linear(256, 10),
            snn.Leaky(beta=snn_beta, threshold=snn_thresh)
        )

    def forward(self, x, num_steps=50):
        mem1 = self.features[2].init_leaky()
        mem2 = self.features[5].init_leaky()
        mem3 = self.features[8].init_leaky()
        mem4 = self.classifier[1].init_leaky()
        mem5 = self.classifier[3].init_leaky()

        spk_rec = []

        for step in range(num_steps):
            cur1 = self.features[1](self.features[0](x))
            spk1, mem1 = self.features[2](cur1, mem1)

            cur2 = self.features[4](self.features[3](spk1))
            spk2, mem2 = self.features[5](cur2, mem2)

            cur3 = self.features[7](self.features[6](spk2))
            spk3, mem3 = self.features[8](cur3, mem3)

            flat = spk3.view(spk3.size(0), -1)

            cur4 = self.classifier[0](flat)
            spk4, mem4 = self.classifier[1](cur4, mem4)

            cur5 = self.classifier[2](spk4)
            spk5, mem5 = self.classifier[3](cur5, mem5)

            spk_rec.append(spk5)

        return torch.stack(spk_rec)

snn_net = ConvertedSNN().to(device)

In [23]:
print("Executing ANN-to-SNN Weight Transfer...")
snn_net.load_state_dict(cnn_net.state_dict(), strict=False)

print("Evaluating SNN on Test Data for Hardware Metrics...")
snn_net.eval()
correct = 0
total = 0
total_spikes = 0
total_possible_spikes = 0

num_steps = 50

with torch.no_grad():
    for data, targets in test_loader:
        data, targets = data.to(device), targets.to(device)

        spk_out = snn_net(data, num_steps=num_steps)

        _, predicted = spk_out.sum(dim=0).max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()

        total_spikes += spk_out.sum().item()
        total_possible_spikes += spk_out.numel()

final_acc = 100. * correct / total
firing_rate = total_spikes / total_possible_spikes

ann_energy = total_possible_spikes * 4.6
snn_energy = total_spikes * 0.9
energy_savings = 100 - (snn_energy / ann_energy * 100)

print("\n=== FINAL NEUROMORPHIC HARDWARE REPORT ===")
print(f"SNN Test Accuracy:     {final_acc:.2f}%")
print(f"Network Firing Rate:   {firing_rate * 100:.2f}% (Sparsity Validated!)")
print(f"Estimated ANN Energy:  {ann_energy:.2f} pJ")
print(f"Estimated SNN Energy:  {snn_energy:.2f} pJ")
print(f"Total Energy Saved:    {energy_savings:.2f}%\n")

Executing ANN-to-SNN Weight Transfer...
Evaluating SNN on Test Data for Hardware Metrics...

=== FINAL NEUROMORPHIC HARDWARE REPORT ===
SNN Test Accuracy:     51.04%
Network Firing Rate:   12.25% (Sparsity Validated!)
Estimated ANN Energy:  12364800.00 pJ
Estimated SNN Energy:  296352.90 pJ
Total Energy Saved:    97.60%

